In [ ]:
!python -m spacy download pt_core_news_sm -q


In [ ]:
!git clone https://github.com/gustavoFraiz/GuaraniTXT.git

In [ ]:
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import textwrap

# Função auxiliar para formatar e imprimir os passos intermediários
def print_step(title, content=""):
    """Imprime os passos intermediários de forma organizada."""
    print("\n" + "="*80)
    print(f"## {title.upper()} ##")
    print("="*80)
    if content:
        # Usando textwrap para formatar textos longos
        if isinstance(content, list):
            for i, item in enumerate(content):
                print(f"Item {i+1}:\n{textwrap.fill(str(item), width=80)}\n")
        else:
            print(textwrap.fill(str(content), width=80))
    print()

In [ ]:
def load_and_clean_text(filepath):
    """Carrega o texto, remove tags e normaliza."""
    try:
        with open(filepath, 'r', encoding='utf-8') as file:
            text = file.read()

        text = re.sub(r'/', '', text)
        text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()

        return text
    except FileNotFoundError:
        print(f"ERRO: O arquivo '{filepath}' não foi encontrado.")
        print("Por favor, faça o upload do arquivo 'O guarani.txt' para o ambiente do Colab.")
        return None

# Execução do Passo 1
print_step("PASSO 1: RECUPERAÇÃO DO LIVRO E LIMPEZA INICIAL")
FILE_PATH = '/content/GuaraniTXT/O guarani.txt'
raw_text = load_and_clean_text(FILE_PATH)

if raw_text:
    print("Texto carregado e limpo com sucesso.")
    print_step("Amostra do Texto Limpo", raw_text[:500] + "...")

In [ ]:
def create_chunks(text, chunk_size=200, overlap_ratio=0.5):
    """Segmenta o texto em chunks com sobreposição."""
    tokens = nltk.word_tokenize(text, language='portuguese')
    overlap = int(chunk_size * overlap_ratio)
    step = chunk_size - overlap

    return [' '.join(tokens[i:i + chunk_size]) for i in range(0, len(tokens), step) if len(tokens[i:i + chunk_size]) > 0]

# Execução do Passo 2
print_step("PASSO 2: SEGMENTAÇÃO EM CHUNKS")
original_chunks = create_chunks(raw_text)
print(f"Texto dividido em {len(original_chunks)} chunks.")
print_step("Amostra dos 2 primeiros Chunks Gerados", original_chunks[:2])

In [ ]:
# Carrega o modelo spaCy
nlp = spacy.load('pt_core_news_lg')

def preprocess_text(text, nlp_model):
    """Lematiza e remove stopwords de um texto."""
    doc = nlp_model(text)
    lemmas = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
    return ' '.join(lemmas)

def vectorize_chunks(chunks, nlp_model):
    """Vetoriza os chunks usando TF-IDF após o pré-processamento."""
    print_step("3.1: Pré-processamento dos Chunks (Lematização e Remoção de Stopwords)")
    processed_chunks = [preprocess_text(chunk, nlp_model) for chunk in chunks]
    print_step("Amostra de Chunk Pré-processado", processed_chunks[0])

    print_step("3.2: Vetorização dos Chunks com TF-IDF")
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(processed_chunks)

    print(f"Matriz TF-IDF criada com shape: {tfidf_matrix.shape}")
    print_step("Dimensões da Matriz", f"Linhas (Chunks): {tfidf_matrix.shape[0]}, Colunas (Vocabulário): {tfidf_matrix.shape[1]}")
    return vectorizer, tfidf_matrix

# Execução do Passo 3
vectorizer, tfidf_matrix = vectorize_chunks(original_chunks, nlp)

In [ ]:
def get_intent_and_entities(query, nlp_model):
    """Detecta a intenção e as entidades da pergunta."""
    doc = nlp_model(query)
    intent = 'desconhecida'
    # Detecção de intenção
    if any(palavra in query.lower() for palavra in ['quem', 'personagem']): intent = 'quem'
    elif any(palavra in query.lower() for palavra in ['onde', 'lugar']): intent = 'onde'
    elif any(palavra in query.lower() for palavra in ['quando', 'época']): intent = 'quando'
    elif any(palavra in query.lower() for palavra in ['como', 'forma', 'maneira']): intent = 'como'
    elif any(palavra in query.lower() for palavra in ['o que', 'qual']): intent = 'o que'

    # Extração de entidades
    entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ in ['PER', 'LOC', 'ORG']]
    return intent, entities

def generate_response(intent, entities, retrieved_chunks):
    """Gera uma resposta formatada usando templates."""
    if not retrieved_chunks:
        return "Desculpe, não encontrei informações relevantes sobre isso no texto de 'O Guarani'."

    context_text = " (...) ".join(retrieved_chunks)

    # Usa templates baseados na intenção
    if intent == 'quem' and entities:
        entity_name = entities[0][0]
        return f"Sobre {entity_name}, a obra relata o seguinte:\n\n\"...{context_text}...\""
    elif intent == 'onde' and entities:
        entity_name = entities[0][0]
        return f"Em relação a {entity_name}, o texto descreve:\n\n\"...{context_text}...\""
    else:
        return f"Encontrei as seguintes informações relevantes no texto:\n\n\"...{context_text}...\""

print("✅ Funções de análise e geração de resposta prontas.")

In [ ]:
print("\n" + "#"*80)
print("## CHATBOT 'O GUARANI' PRONTO PARA USO. DIGITE 'sair' PARA ENCERRAR. ##")
print("#"*80 + "\n")

while True:
    query = input(">> Você: ")
    if query.lower() == 'sair':
        print(">> Chatbot: Até logo!")
        break

    # --- Início do fluxo de resposta por consulta ---
    print_step("Processando sua pergunta...")

    # Passo 4.1: Pré-processamento da Pergunta
    processed_query = preprocess_text(query, nlp)
    print_step("PASSO 4.1: PRÉ-PROCESSAMENTO DA PERGUNTA", f"Original: '{query}'\nProcessada: '{processed_query}'")

    # Passo 4.2: Análise de Intenção e Entidades
    intent, entities = get_intent_and_entities(query, nlp)
    print_step("PASSO 4.2: ANÁLISE DE INTENÇÃO E ENTIDADES", f"Intenção Detectada: {intent}\nEntidades Encontradas: {entities}")

    # Passo 4.3: Vetorização da Pergunta
    query_vector = vectorizer.transform([processed_query])
    print_step("PASSO 4.3: VETORIZAÇÃO DA BUSCA (VETOR TF-IDF DA PERGUNTA)", f"Vetor da pergunta possui {query_vector.nnz} termos relevantes.")

    # Passo 4.4: Busca por Similaridade
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    threshold = 0.1 # Limiar mínimo de similaridade
    top_indices = np.argsort(similarities)[-3:][::-1] # Pega os 3 mais relevantes

    retrieved_chunks_for_response = []
    log_output = []
    for idx in top_indices:
        if similarities[idx] > threshold:
            retrieved_chunks_for_response.append(original_chunks[idx])
            log_output.append(f"Chunk {idx} (Similaridade: {similarities[idx]:.4f}):\n{original_chunks[idx]}")

    print_step("PASSO 4.4: RECUPERAÇÃO DE TEXTOS (TOP-3 CHUNKS RELEVANTES)", log_output if log_output else "Nenhum chunk relevante encontrado acima do limiar.")

    # Passo 5: Geração da Resposta
    final_response = generate_response(intent, entities, retrieved_chunks_for_response)
    print_step("PASSO 5: RESPOSTA FINAL DO SISTEMA", final_response)